# NWB + DANDI: a first look

**Goal:** by the end of this notebook you will know what an NWB file is, how DANDI organizes them, and how to open one straight from the archive and pull out spikes and trials — without downloading anything to disk.

**Runtime:** standard Colab CPU runtime, ~10 minutes.

## The two things you're learning

| | What it is | Analogy |
|---|---|---|
| **NWB** (Neurodata Without Borders) | A *file format + schema* for neurophysiology data. One `.nwb` file = one recording session. Under the hood it's HDF5 (think: a filesystem inside a file) with strict rules about what goes where and what metadata is required. | A standardized lab notebook page |
| **DANDI** | A *public archive* that stores NWB files (and BIDS/Zarr). Datasets are called **Dandisets**, each with a 6-digit ID like `000006`. Files inside a Dandiset are **assets**. Everything sits on S3, so you can stream pieces of a file over HTTP. | The library that holds the notebooks |

The key design idea of NWB: data and the metadata needed to interpret it live in the **same file**. Spike times, the electrode they came from, the trial structure, and the animal's species all travel together.

## Anatomy of an `NWBFile`

```
NWBFile
├── session_description, session_start_time, identifier   (required metadata)
├── subject            → species, age, sex, ...
├── devices / electrode_groups / electrodes               (where the signals came from)
├── acquisition        → raw data as it was recorded (TimeSeries, ElectricalSeries, ...)
├── stimulus           → what was presented
├── processing         → derived data (filtered LFP, dF/F, position, ...)
├── intervals.trials   → a table, one row per trial
└── units              → a table, one row per sorted neuron, with a ragged 'spike_times' column
```

Two container types do most of the work: **TimeSeries** (data + timestamps or a rate) and **DynamicTable** (a pandas-like table that can hold ragged columns). `trials`, `units`, and `electrodes` are all DynamicTables.

## 0. Install

Three packages: `pynwb` (read/write NWB), `dandi` (talk to the archive), `remfile` (stream HDF5 over HTTP). Takes ~1 minute on Colab.

In [ ]:
%pip install -q pynwb dandi remfile

import pynwb, dandi
print("pynwb", pynwb.__version__, "| dandi", dandi.__version__)

## 1. Find the file on DANDI

We'll use **Dandiset 000006** — Economo & Svoboda, extracellular recordings from mouse anterior lateral motor cortex (ALM) during a delayed-response licking task. It's the example the PyNWB docs use because the files are tiny.

A Dandiset has **versions**: `draft` (editable) and frozen published versions like `0.220126.1855`. For reproducible analysis you'd pin a published version; for poking around, `draft` is fine.

The `DandiAPIClient` is a thin wrapper over DANDI's REST API.

In [ ]:
from dandi.dandiapi import DandiAPIClient

DANDISET_ID = "000006"

client = DandiAPIClient()
dandiset = client.get_dandiset(DANDISET_ID, "draft")

meta = dandiset.get_raw_metadata()
print("Name:       ", meta["name"])
print("License:    ", meta.get("license"))
print("Contributors:", [c["name"] for c in meta.get("contributor", []) if "name" in c][:4], "...")
print()
print("Web page:   ", f"https://dandiarchive.org/dandiset/{DANDISET_ID}")

Each file in a Dandiset is an **asset**. DANDI enforces a naming convention: `sub-<subject>/sub-<subject>_ses-<session>.nwb`. Let's list the first few and their sizes.

In [ ]:
assets = list(dandiset.get_assets())
print(f"{len(assets)} assets in this dandiset\n")

for a in sorted(assets, key=lambda a: a.size)[:8]:
    print(f"{a.size/1e6:7.2f} MB   {a.path}")

## 2. Get a URL and stream the file

We won't download the file. Instead we ask DANDI for the asset's S3 URL, then open it with `remfile`, which fetches only the byte ranges HDF5 asks for. `h5py` sees a normal file; `pynwb` sits on top of `h5py`.

Why this matters: many NWB files are tens of GB (raw voltage traces). Streaming lets you read a 2 MB `units` table from a 50 GB file in seconds.

In [ ]:
import h5py
import remfile
from pynwb import NWBHDF5IO

FILEPATH = "sub-anm372795/sub-anm372795_ses-20170718.nwb"   # ~450 kB

asset = dandiset.get_asset_by_path(FILEPATH)
s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
print(s3_url)

# Open the remote file. We keep these handles open for the rest of the notebook.
rem_file = remfile.File(s3_url)
h5_file = h5py.File(rem_file, "r")
io = NWBHDF5IO(file=h5_file, load_namespaces=True)   # load_namespaces: read schema extensions stored in the file
nwb = io.read()

nwb   # pynwb pretty-prints the file's top-level structure

Read that printout against the anatomy diagram above. You should see `subject`, `electrodes`, `electrode_groups`, `trials`, and `units`. Note what's *absent*: there's no `acquisition` with raw voltage — this dataset shares only sorted spikes, which is common.

## 3. Session and subject metadata

These are plain attributes.

In [ ]:
print("Session:    ", nwb.session_description)
print("Start time: ", nwb.session_start_time)
print("Identifier: ", nwb.identifier)
print("Institution:", nwb.institution)
print()
s = nwb.subject
print("Subject:", s.subject_id, "|", s.species, "|", s.sex, "| age:", s.age, "| genotype:", s.genotype)

## 4. Tables: `trials`, `electrodes`, `units`

Any `DynamicTable` converts to a pandas DataFrame with `.to_dataframe()`. The `trials` table has required `start_time` / `stop_time` columns; everything else is experiment-specific, so **always print the column descriptions** — that's where the experimenters tell you what the values mean.

In [ ]:
trials = nwb.trials.to_dataframe()
print(trials.shape)

for col in nwb.trials.colnames:
    print(f"  {col:20s} {nwb.trials[col].description}")

trials.head()

In [ ]:
electrodes = nwb.electrodes.to_dataframe()
print(electrodes.shape)
electrodes.head()

The `units` table is the one you'll use most. Its `spike_times` column is **ragged** — each neuron has a different number of spikes — so pandas shows it as a column of arrays. When streaming, index a single unit (`nwb.units["spike_times"][i]`) rather than converting the whole table, so you only pull the bytes you need.

In [ ]:
print("Units columns:", nwb.units.colnames)
print("Number of units:", len(nwb.units))
print()

units = nwb.units.to_dataframe()   # small file, so this is fine here
units.head()

## 5. A minimal analysis: trial-aligned raster

Everything needed is already in hand: spike times (in seconds, on the session clock) and trial start times (same clock). Aligning them is a subtraction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

UNIT = 0                       # try others
WINDOW = (-1.0, 3.0)           # seconds relative to trial start

spikes = np.asarray(nwb.units["spike_times"][UNIT])
print(f"unit {UNIT}: {len(spikes)} spikes, mean rate {len(spikes)/(spikes.max()-spikes.min()):.2f} Hz")

# Pick an alignment event: use a cue time if the trials table has one, else trial start
align_col = next((c for c in trials.columns if "cue" in c.lower() and "time" in c.lower()), "start_time")
align = trials[align_col].to_numpy()
print("aligning to:", align_col)

fig, (ax_r, ax_p) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                                 gridspec_kw={"height_ratios": [3, 1]})
all_rel = []
for i, t0 in enumerate(align):
    rel = spikes[(spikes >= t0 + WINDOW[0]) & (spikes < t0 + WINDOW[1])] - t0
    ax_r.plot(rel, np.full_like(rel, i), "|", color="k", ms=3)
    all_rel.append(rel)
ax_r.axvline(0, color="C3", lw=1)
ax_r.set_ylabel("trial")
ax_r.set_title(f"{nwb.subject.subject_id} — unit {UNIT}")

bins = np.arange(WINDOW[0], WINDOW[1] + 0.05, 0.05)
counts, _ = np.histogram(np.concatenate(all_rel), bins)
ax_p.bar(bins[:-1], counts / len(align) / 0.05, width=0.05, color="C0", align="edge")
ax_p.axvline(0, color="C3", lw=1)
ax_p.set_xlabel(f"time from {align_col} (s)")
ax_p.set_ylabel("rate (Hz)")
plt.tight_layout()

**Exercise:** the trials table has a column describing the trial type (left vs. right lick, or similar — check the descriptions you printed in §4). Split the raster by that column. If you see a unit that fires differently before the two response directions, you've reproduced the core finding of the paper on a laptop in five lines.

Close the file when done (it releases the HTTP connection).

In [ ]:
io.close()
h5_file.close()

## 6. Write your own NWB file

Reading is half the story. Writing a tiny file makes the schema concrete: you'll see which fields are *required* and how a `TimeSeries` is built.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pynwb import NWBFile, TimeSeries
from pynwb.file import Subject

nwbfile = NWBFile(
    session_description="toy session: synthetic sine wave",      # required
    identifier="toy-0001",                                        # required, unique per file
    session_start_time=datetime(2026, 1, 1, 9, 0, tzinfo=ZoneInfo("America/Los_Angeles")),  # required, tz-aware
    experimenter="You",
    lab="Colab",
)
nwbfile.subject = Subject(subject_id="virtual-01", species="Mus musculus", sex="U", age="P90D")

rate = 100.0
t = np.arange(0, 10, 1 / rate)
ts = TimeSeries(
    name="synthetic_signal",
    data=np.sin(2 * np.pi * 1.0 * t),
    unit="V",
    starting_time=0.0,
    rate=rate,
    description="1 Hz sine, pretend it's a voltage",
)
nwbfile.add_acquisition(ts)

# Trials: add the columns you want, then rows
nwbfile.add_trial_column(name="condition", description="which condition this trial belongs to")
for i, start in enumerate(np.arange(0, 10, 2.0)):
    nwbfile.add_trial(start_time=start, stop_time=start + 1.5, condition="A" if i % 2 == 0 else "B")

with NWBHDF5IO("toy.nwb", "w") as io_w:
    io_w.write(nwbfile)

print("wrote toy.nwb")

Read it back — same API as the remote file, just a local path. Then look at the raw HDF5 to see there's no magic.

In [ ]:
with NWBHDF5IO("toy.nwb", "r") as io_r:
    back = io_r.read()
    print(back.acquisition["synthetic_signal"].data[:5], back.acquisition["synthetic_signal"].rate)
    display(back.trials.to_dataframe())

# Under the hood: plain HDF5 groups and datasets
with h5py.File("toy.nwb", "r") as f:
    f.visit(lambda name: print(name) if name.count("/") < 2 else None)

## 7. Validate

NWB ships a validator that checks a file against the schema. Run it on anything you write before sharing (DANDI runs it on upload too).

In [ ]:
!pynwb-validate toy.nwb

## What you now know

- An **NWB file** is HDF5 with a schema: required session metadata, a `subject`, tables for `electrodes` / `trials` / `units`, and `TimeSeries` containers for continuous data.
- **DANDI** stores NWB files as assets in versioned Dandisets, addressable by ID + path, backed by S3.
- `dandi` → get a URL; `remfile` + `h5py` → stream it; `pynwb` → read it with semantics. `.to_dataframe()` gets you to pandas.
- Writing a file forces you to supply the required metadata, which is the whole point of the standard.

## Where to go next

- **Browse Dandisets:** https://dandiarchive.org — filter by species/modality. Change `DANDISET_ID` and `FILEPATH` above and re-run; most of this notebook will work unchanged.
- **Neurosift** (https://neurosift.app) renders any DANDI NWB file in the browser — useful for deciding what to load before writing code.
- **PyNWB tutorials:** https://pynwb.readthedocs.io/en/stable/tutorials/ — especially *NWB File Basics* and *Extracellular Electrophysiology*.
- **NWB Overview:** https://nwb-overview.readthedocs.io — the conceptual docs, including the full list of neurodata types.
- **Bigger files:** for multi-GB files, read slices (`data[0:1000, :]`) rather than `[:]`, and look into `lindi` or Zarr-backed NWB for faster remote access.
- **Converting your own data:** `NeuroConv` (https://neuroconv.readthedocs.io) has interfaces for most acquisition systems.
